In [1]:
#!pip install biopython pandas matplotlib seaborn pyarrow openpyxl umap-learn plotly

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="umap")

In [4]:
from amr.normalisation import recompute_resid_log_bg_by_antibiotic_level
from amr.predictive import (
    baseline_global_average,
    baseline_persistence,
    build_amr_with_embeddings,
    build_data_and_target,
    build_taxonomy_onehot,
    evaluate_predictions,
    leave_one_pathogen_out_with_embeddings,
    random_forest_experiment_with_embeddings,
    rolling_origin_experiment_with_embeddings,
)


---

In [5]:
antibiotic_meta_df = pd.read_excel("antibiotic_list2.xlsx")
antibiotics_list = antibiotic_meta_df['Antibiotic'].astype(str).values
print(f'#antibiotics in metadata: {len(antibiotics_list)}')

data_df = pd.read_parquet("data.parquet")
pathogens_list = data_df[data_df['n_causation']>=3]['CanonicalSpecies'].values
data_df, taxonomy_onehot = build_taxonomy_onehot(data_df)
print(f'#pathogens: {len(pathogens_list)}')


#antibiotics in metadata: 245
#pathogens: 773


In [6]:
data_df

,Genus,CanonicalSpecies,n_causation,n_articles,n_sampled,pathogenic_index,embedding,anonymised_embedding,onehot_embedding,domain,kingdom,phylum,class,order,family,taxonomy_embeddings
0,Abiotrophia,Abiotrophia defectiva,6,6,6,0.160939,"[0.018194107338786125, -0.004734894765230517, ...","[0.014040807844139636, -0.028420374651129048, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.166...",Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Aerococcaceae,"[0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
1,Acetobacter,Acetobacter indonesiensis,1,3,3,0.000000,"[-0.021512236446142197, -0.03992129862308502, ...","[-0.0028289835900068283, -0.04833690822124481,...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodospirillales,Acetobacteraceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
2,Acetobacter,Acetobacter tropicalis,1,2,2,0.000000,"[-0.0070418259128928185, -0.005829853471368551...","[0.005147432442754507, -0.013049994595348835, ...","[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodospirillales,Acetobacteraceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
3,Achromobacter,Achromobacter animicus,1,2,2,0.000000,"[0.011074555106461048, 0.013332720845937729, 0...","[0.012643104419112206, -0.01178667601197958, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Alcaligenaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
4,Achromobacter,Achromobacter denitrificans,75,36,36,0.387804,"[0.014773702074987037, -0.004635852294159122, ...","[0.011467029051855206, -0.012983008560258896, ...","[0.04, 0.013333333333333334, 0.066666666666666...",Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Alcaligenaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1274,Yersinia,Yersinia enterocolitica,85,3419,200,0.654031,"[-0.009990866917788105, 0.009774883079979405, ...","[0.0024005417033656747, -0.009907858140479005,...","[0.0, 0.0, 0.0, 0.03529411764705882, 0.0, 0.0,...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1275,Yersinia,Yersinia frederiksenii,1,21,21,0.000000,"[-0.00901515781879425, 0.02502085641026497, 0....","[0.01804814115166664, 0.005011445842683315, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1276,Yersinia,Yersinia pestis,124,2184,200,0.647694,"[-0.006688895473886195, 0.014983039929487154, ...","[0.011833864605177364, -0.0048193380043226024,...","[0.0, 0.0, 0.0, 0.016129032258064516, 0.0, 0.0...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1277,Yersinia,Yersinia pseudotuberculosis,88,1234,200,0.565611,"[-0.01641193700528344, 0.008593090970448842, 0...","[0.004612311640316461, -0.005791128928070644, ...","[0.011363636363636364, 0.0, 0.0113636363636363...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."


In [7]:
PREDICTION_MODE = "resid_log_bg"  # choose: "count" or "resid_log_bg"
ANTIBIOTIC_LEVEL = "antibiotic"  # choose: "antibiotic", "class", "simplified_category", "high_level_category", "origin"
ERROR_ON_UNMAPPED = True

if PREDICTION_MODE == "count":
    AMR_PARQUET = "pmids_results.parquet"
    VALUE_COL = "count"
    APPLY_LOG1P = True
    PLOT_XLABEL = "Count"
    SAMPLE_WEIGHT_MODE = "count_if_available"
    amr_df = pd.read_parquet(AMR_PARQUET)
elif PREDICTION_MODE == "resid_log_bg":
    AMR_PARQUET = "amr_df_normalised.parquet"
    VALUE_COL = "resid_log_bg"
    APPLY_LOG1P = False
    PLOT_XLABEL = "resid_log_bg"
    SAMPLE_WEIGHT_MODE = "none"
    amr_base_df = pd.read_parquet(AMR_PARQUET)
    amr_df = recompute_resid_log_bg_by_antibiotic_level(
        amr_base_df,
        antibiotic_level=ANTIBIOTIC_LEVEL,
        antibiotic_meta_df=antibiotic_meta_df,
        count_col="count",
        error_on_unmapped=ERROR_ON_UNMAPPED,
        validate_bg_consistency=False,
    )
else:
    raise ValueError(f"Unsupported PREDICTION_MODE: {PREDICTION_MODE}")

RESULTS_PARQUET = f"results_{PREDICTION_MODE}_{ANTIBIOTIC_LEVEL}.parquet"

print(f"Mode: {PREDICTION_MODE} | value_col={VALUE_COL} | apply_log1p={APPLY_LOG1P}")
print(f"Antibiotic level: {ANTIBIOTIC_LEVEL} | error_on_unmapped={ERROR_ON_UNMAPPED}")
print(f"Sample weight mode: {SAMPLE_WEIGHT_MODE}")
print(f"AMR source: {AMR_PARQUET}")
print(f"LOPO results file: {RESULTS_PARQUET}")
amr_df


Mode: resid_log_bg | value_col=resid_log_bg | apply_log1p=False
Antibiotic level: antibiotic | error_on_unmapped=True
Sample weight mode: none
AMR source: amr_df_normalised.parquet
LOPO results file: results_resid_log_bg_antibiotic.parquet


,pathogen,antibiotic,start_date,end_date,count,bg_antibiotic_n,bg_pathogen_n,frac_among_pathogen,frac_among_antibiotic,geo_frac,expected_bg,resid_log_bg,z_bg,antibiotic_level
0,Abiotrophia defectiva,Cefepime,2015/01/01,2019/12/31,1,202,6,0.285714,0.009852,0.053056,1.000000,5.000000e-10,1.000000e-09,antibiotic
1,Abiotrophia defectiva,Cefotaxime,2015/01/01,2019/12/31,1,200,17,0.111111,0.009950,0.033250,1.000000,5.000000e-10,1.000000e-09,antibiotic
2,Abiotrophia defectiva,Ceftriaxone,2010/01/01,2014/12/31,1,229,8,0.222222,0.008696,0.043959,1.000000,5.000000e-10,1.000000e-09,antibiotic
3,Abiotrophia defectiva,Ciprofloxacin,2010/01/01,2014/12/31,1,192,6,0.285714,0.010363,0.054413,1.000000,5.000000e-10,1.000000e-09,antibiotic
4,Abiotrophia defectiva,Clindamycin,2000/01/01,2004/12/31,1,252,8,0.222222,0.007905,0.041913,1.055633,-2.743656e-02,-5.414703e-02,antibiotic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24044,Yokenella regensburgei,Fusidic-acid,2000/01/01,2004/12/31,1,12,517,0.003861,0.153846,0.024372,1.000000,5.000000e-10,1.000000e-09,antibiotic
24045,Yokenella regensburgei,Linezolid,2000/01/01,2004/12/31,1,163,111,0.017857,0.012195,0.014757,1.000000,5.000000e-10,1.000000e-09,antibiotic
24046,Yokenella regensburgei,Loracarbef,2000/01/01,2004/12/31,1,178,123,0.016129,0.011173,0.013424,1.000000,5.000000e-10,1.000000e-09,antibiotic
24047,Yokenella regensburgei,Oxacillin,2000/01/01,2004/12/31,1,226,116,0.017094,0.008811,0.012272,1.000000,5.000000e-10,1.000000e-09,antibiotic


---

In [8]:
X, Y, feature_df, target_df = build_data_and_target(
    amr_df,
    value_col=VALUE_COL,
    apply_log1p=APPLY_LOG1P,
    antibiotic_level=ANTIBIOTIC_LEVEL,
    antibiotic_meta_df=antibiotic_meta_df,
    error_on_unmapped=ERROR_ON_UNMAPPED,
)

r2_persist, rmse_persist = baseline_persistence(feature_df, target_df)
r2_avg, rmse_avg = baseline_global_average(target_df)
print(f"Persistence baseline: R²={r2_persist:.3f}, RMSE={rmse_persist:.3f}")
print(f"Global average baseline: R²={r2_avg:.3f}, RMSE={rmse_avg:.3f}")

# Per-antibiotic diagnostics (MAE/RMSE/R²)
antibiotics = [c.replace("_target", "") for c in target_df.columns]

# Persistence predictions
persist_preds = []
for ab in target_df.columns:
    ab_name = ab.replace("_target", "")
    last_cols = [c for c in feature_df.columns if c.startswith(ab_name + "_")]
    if last_cols:
        persist_preds.append(feature_df[sorted(last_cols)[-1]].values)
    else:
        persist_preds.append(np.zeros(len(feature_df), dtype=float))
Y_pred_persist = np.vstack(persist_preds).T

persist_overall, persist_per_ab = evaluate_predictions(
    target_df.to_numpy(),
    Y_pred_persist,
    antibiotics=antibiotics,
)
print("Persistence overall:", persist_overall)
persist_per_ab.sort_values("RMSE", ascending=False).head(10)


Persistence baseline: R²=-19219834203310780.000, RMSE=0.160
Global average baseline: R²=0.000, RMSE=0.171
Persistence overall: {'R2': -1.921983420331078e+16, 'RMSE': 0.16006765913085103, 'MAE': 0.027471290201294327}


,antibiotic,R2,RMSE,MAE
146,Tetracycline,-0.016154,0.517659,0.241398
60,Erythromycin,0.268779,0.392395,0.190853
70,Gentamicin,-1.267800,0.382314,0.138410
41,Chloramphenicol,-1.001386,0.375764,0.120104
3,Ampicillin,0.503232,0.367681,0.176526
33,Ceftazidime,0.094254,0.360106,0.095580
43,Ciprofloxacin,0.102394,0.351063,0.156955
153,Trimethoprim,0.356891,0.349461,0.118998
1,Amoxicillin,-0.316547,0.341367,0.111523
135,Sulfamethoxazole,0.348606,0.334713,0.115045


In [9]:
%%time
# Secondary benchmark: random pathogen split
results, avg_scores = random_forest_experiment_with_embeddings(
    amr_df,
    data_df,
    embedding_cols=None,
    value_col=VALUE_COL,
    apply_log1p=APPLY_LOG1P,
    n_repeats=5,
    n_estimators=300,
    max_depth=None,
    antibiotic_level=ANTIBIOTIC_LEVEL,
    antibiotic_meta_df=antibiotic_meta_df,
    error_on_unmapped=ERROR_ON_UNMAPPED,
    sample_weight_mode=SAMPLE_WEIGHT_MODE,
)
avg_scores


Repeat 1/5: MAE=0.024, RMSE=0.129, R²=-594891921814800.125
Repeat 2/5: MAE=0.035, RMSE=0.172, R²=-2188729747112046.250
Repeat 3/5: MAE=0.026, RMSE=0.142, R²=-711745732295001.750
Repeat 4/5: MAE=0.029, RMSE=0.159, R²=-1486062025147489.750
Repeat 5/5: MAE=0.019, RMSE=0.108, R²=-1013167490989897.750

Average Performance:
  MAE  = 0.026 ± 0.005
  RMSE = 0.142 ± 0.022
  R²   = -1198919383471847.250 ± 582900773551634.500
CPU times: user 1h 33min 47s, sys: 2min 47s, total: 1h 36min 35s
Wall time: 18min 16s


{'R2_mean': -1198919383471847.2,
 'R2_std': 582900773551634.5,
 'RMSE_mean': 0.1419534779057678,
 'RMSE_std': 0.022313448789944394,
 'MAE_mean': 0.026463143111755943,
 'MAE_std': 0.00522954170664162}

In [8]:
%%time
# Secondary benchmark with embeddings: random pathogen split
results_embeddings, avg_scores_embeddings = random_forest_experiment_with_embeddings(
    amr_df,
    data_df,
    embedding_cols=["taxonomy_embeddings", "anonymised_embedding"],
    value_col=VALUE_COL,
    apply_log1p=APPLY_LOG1P,
    n_repeats=5,
    n_estimators=300,
    max_depth=None,
    antibiotic_level=ANTIBIOTIC_LEVEL,
    antibiotic_meta_df=antibiotic_meta_df,
    error_on_unmapped=ERROR_ON_UNMAPPED,
    sample_weight_mode=SAMPLE_WEIGHT_MODE,
)
avg_scores_embeddings


Repeat 1/5: MAE=0.026, RMSE=0.131, R²=-723598880968679.625
Repeat 2/5: MAE=0.037, RMSE=0.173, R²=-1548652881892947.000
Repeat 3/5: MAE=0.027, RMSE=0.143, R²=-663176106264679.125
Repeat 4/5: MAE=0.030, RMSE=0.160, R²=-1290222679470752.000
Repeat 5/5: MAE=0.021, RMSE=0.111, R²=-854993508630310.000

Average Performance:
  MAE  = 0.028 ± 0.005
  RMSE = 0.144 ± 0.022
  R²   = -1016128811445473.625 ± 344912537385215.250
CPU times: user 1d 5h 36min 49s, sys: 18min 48s, total: 1d 5h 55min 37s
Wall time: 2h 26min 8s


{'R2_mean': -1016128811445473.6,
 'R2_std': 344912537385215.25,
 'RMSE_mean': 0.1438434706058737,
 'RMSE_std': 0.02173094356125481,
 'MAE_mean': 0.02799362768568034,
 'MAE_std': 0.005321631318942619}

In [ ]:
%%time
# Primary benchmark: rolling-origin + LOPO (harder generalization)
rolling_results, rolling_summary = rolling_origin_experiment_with_embeddings(
    amr_df,
    data_df,
    embedding_cols=["taxonomy_embeddings", "anonymised_embedding"],
    value_col=VALUE_COL,
    apply_log1p=APPLY_LOG1P,
    n_repeats=3,
    n_estimators=300,
    max_depth=None,
    antibiotic_level=ANTIBIOTIC_LEVEL,
    antibiotic_meta_df=antibiotic_meta_df,
    error_on_unmapped=ERROR_ON_UNMAPPED,
    split_strategy="lopo",
    sample_weight_mode=SAMPLE_WEIGHT_MODE,
)

print("Rolling-origin summary:", rolling_summary)
rolling_results.groupby("target_bin")[["MAE", "RMSE", "R2"]].mean().sort_index()


Done pathogen 1/77: Achromobacter xylosoxidans
Done pathogen 2/77: Acinetobacter calcoaceticus
Done pathogen 3/77: Actinomyces viscosus
Done pathogen 4/77: Aeromonas hydrophila
Done pathogen 5/77: Aggregatibacter actinomycetemcomitans
Done pathogen 6/77: Bacillus cereus
Done pathogen 7/77: Bacillus licheniformis
Done pathogen 8/77: Bacteroides fragilis
Done pathogen 9/77: Bordetella bronchiseptica
Done pathogen 10/77: Bordetella pertussis
Done pathogen 11/77: Burkholderia cepacia
Done pathogen 12/77: Burkholderia pseudomallei
Done pathogen 13/77: Campylobacter fetus
Done pathogen 14/77: Campylobacter jejuni
Done pathogen 15/77: Chlamydia psittaci
Done pathogen 16/77: Citrobacter freundii
Done pathogen 17/77: Citrobacter koseri
Done pathogen 18/77: Clostridioides difficile
Done pathogen 19/77: Clostridium perfringens
Done pathogen 20/77: Corynebacterium diphtheriae
Done pathogen 21/77: Corynebacterium xerosis
Done pathogen 22/77: Cutibacterium acnes
Done pathogen 23/77: Eikenella corrod

---

In [ ]:
%%time
RUN_EXPERIMENT = False # Set to False to load existing results

if RUN_EXPERIMENT:
    results_df = leave_one_pathogen_out_with_embeddings(
        amr_df,
        data_df,
        embedding_cols=["taxonomy_embeddings", "anonymised_embedding"],
        value_col=VALUE_COL,
        apply_log1p=APPLY_LOG1P,
        n_estimators=300,
        n_repeats=5,
        antibiotic_level=ANTIBIOTIC_LEVEL,
        antibiotic_meta_df=antibiotic_meta_df,
        error_on_unmapped=ERROR_ON_UNMAPPED,
    )
    results_df.to_parquet(RESULTS_PARQUET, index=False)
else:
    results_df = pd.read_parquet(RESULTS_PARQUET)

if "value_col" in results_df.columns:
    vals = set(results_df["value_col"].dropna().astype(str).unique().tolist())
    if vals and vals != {VALUE_COL}:
        raise ValueError(f"Loaded {RESULTS_PARQUET} but value_col={vals}; expected {VALUE_COL}")
if "antibiotic_level" in results_df.columns:
    lvls = set(results_df["antibiotic_level"].dropna().astype(str).unique().tolist())
    if lvls and lvls != {ANTIBIOTIC_LEVEL}:
        raise ValueError(f"Loaded {RESULTS_PARQUET} but antibiotic_level={lvls}; expected {ANTIBIOTIC_LEVEL}")

# LOPO aggregate metrics using value columns
lopo_eval = results_df.pivot(index="pathogen", columns="antibiotic", values="actual_value").sort_index()
lopo_pred = results_df.pivot(index="pathogen", columns="antibiotic", values="predicted_value").reindex(lopo_eval.index)
overall_lopo, per_ab_lopo = evaluate_predictions(
    lopo_eval.to_numpy(),
    lopo_pred.to_numpy(),
    antibiotics=lopo_eval.columns.tolist(),
)
print("LOPO overall:", overall_lopo)
per_ab_lopo.sort_values("MAE", ascending=False).head(10)


In [ ]:
import random
import matplotlib.pyplot as plt

# pick one pathogen at random
pathogen = random.choice(list(set(results_df['pathogen'].astype(str))))

# filter rows for that pathogen (exact match)
subset = results_df[results_df["pathogen"] == pathogen].copy()

# drop antibiotics where both values are 0 or NaN
subset = subset[~((subset["actual_value"].fillna(0) == 0) & 
                  (subset["predicted_value"].fillna(0) == 0))]

# plot if any data remains
if not subset.empty:
    subset.set_index("antibiotic")[["actual_value", "predicted_value"]].plot(
        kind="barh",
        figsize=(10, 20),
        title=f"Pathogen: {pathogen}"
    )
    plt.xlabel(PLOT_XLABEL)
    plt.ylabel("Antibiotic")
    plt.tight_layout()
    plt.show()
else:
    print(f"No non-empty values found for pathogen {pathogen}")

results_df[results_df["pathogen"] == pathogen]


---